# MiniCoder Kaggle: Tiny 학습·Output 체크포인트·다른 세션 재개

최종 ZIP을 Kaggle Dataset으로 올려 Notebook input에 연결하고, Notebook settings에서 GPU를 직접 선택하세요. Kaggle API는 별도 노트북을 외부에서 실행하는 API로 여기서 사용하지 않습니다. 계정 quota/session availability는 사용자 계정에서 확인해야 합니다. 학습 output을 commit한 뒤 다음 Notebook version에서 이전 output을 input으로 연결해 재개할 수 있습니다.

In [ ]:
from pathlib import Path
import zipfile, shutil
archives = sorted(Path('/kaggle/input').rglob('*.zip'))
assert archives, 'Attach a Kaggle Dataset containing the final MiniCoder ZIP.'
zip_path = next((p for p in archives if 'minicoder' in p.name.lower()), archives[0])
PROJECT_BASE = Path('/kaggle/working/minicoder_project')
shutil.rmtree(PROJECT_BASE, ignore_errors=True); PROJECT_BASE.mkdir(parents=True)
with zipfile.ZipFile(zip_path) as zf:
    root = PROJECT_BASE.resolve()
    for info in zf.infolist():
        target = (PROJECT_BASE / info.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError(f'Unsafe ZIP member: {info.filename}')
    zf.extractall(PROJECT_BASE)
candidates = [p for p in [PROJECT_BASE, *PROJECT_BASE.iterdir()] if (p/'model'/'config.py').exists()]
assert candidates, 'ZIP must contain model/config.py'
PROJECT = candidates[0]
print('Project:', PROJECT)
%cd $PROJECT

In [ ]:
%pip install -r requirements-training.txt
# For DATASET_MODE="public" only, run: %pip install -r requirements-public.txt
import torch
print('PyTorch:', torch.__version__, 'CUDA runtime:', torch.version.cuda, 'GPU assigned:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), 'VRAM total/free:', torch.cuda.get_device_properties(0).total_memory, torch.cuda.mem_get_info(0)[0])
else:
    print('CPU only. Kaggle GPU quota is not inferred.')

In [ ]:
# Choose synthetic for the smoke path, public for an explicitly licensed Hugging Face dataset,
# or local for a dataset file/directory you own or are licensed to use.
DATASET_MODE = "synthetic"  # synthetic | public | local
SPLIT_OUT = "train"

from pathlib import Path
import json
from dataset.pipeline import DataConfig, build, tokenize_split, update_manifest_token_counts
from dataset.tokenizer import MiniCoderTokenizer
from dataset.registry import DatasetRegistry
from dataset.synthetic import write_smoke_corpus

raw = PROJECT / "raw_dataset"
data = PROJECT / "processed_dataset"
registry = DatasetRegistry(PROJECT / "data" / "dataset_registry.json")
if DATASET_MODE == "synthetic":
    write_smoke_corpus(str(raw), n=160)
    sources = [
        {"path": str(raw / "code"), "name": "synthetic_code", "license": "CC0-1.0"},
        {"path": str(raw / "pairs.jsonl"), "name": "synthetic_pairs", "license": "CC0-1.0"},
    ]
elif DATASET_MODE == "public":
    # Install the optional dependency first: %pip install -r requirements-public.txt
    # Verify the upstream license and official dataset page yourself; values below are examples to replace.
    from dataset.public import load_public_hf_dataset
    DATASET_ID = ""       # set after checking the official dataset card
    DATASET_LICENSE = ""   # set the actual license; no default license is assumed
    DATASET_URL = ""       # official HTTPS dataset card/source page
    assert DATASET_ID and DATASET_LICENSE and DATASET_URL, "Fill verified public dataset metadata first"
    fetched = load_public_hf_dataset(DATASET_ID, raw / "public.jsonl", license_id=DATASET_LICENSE,
                                    source_url=DATASET_URL, split=SPLIT_OUT, text_field="content",
                                    max_samples=20000)
    registry.register("public_source", raw / "public.jsonl", DATASET_LICENSE, DATASET_URL)
    sources = [{"path": str(raw / "public.jsonl"), "name": DATASET_ID, "license": DATASET_LICENSE,
                "source_url": DATASET_URL}]
    print("Public dataset provenance:", json.dumps(fetched, indent=2))
elif DATASET_MODE == "local":
    USER_DATA_PATH = Path("/path/to/your/authorized/dataset")
    USER_DATA_LICENSE = "MIT"  # use the actual license; unknown/non-reviewed licenses are blocked
    entry = registry.register("my_local_dataset", USER_DATA_PATH, USER_DATA_LICENSE,
                              source_url=None, notes="User-supplied dataset")
    sources = [{"path": entry["path"], "name": entry["name"], "license": entry["license"]}]
else:
    raise ValueError("DATASET_MODE must be synthetic, public, or local")

manifest = build(DataConfig(sources=sources, out_dir=str(data), val_ratio=0.1, min_val_samples=5))
texts = [json.loads(line)["text"] for line in (data / "train.jsonl").read_text(encoding="utf-8").splitlines()]
tokenizer = MiniCoderTokenizer(vocab_size=1000)
tokenizer.train_from_iterator(texts)
tokenizer.save(str(PROJECT / "tokenizer.json"))
split_meta = {}
for split in ("train", "val"):
    split_meta[split] = tokenize_split(data / f"{split}.jsonl", tokenizer,
                                       data / f"{split}.bin", manifest["dataset_version"])
manifest = update_manifest_token_counts(data, split_meta)
print("Dataset", manifest["dataset_version"], "tokens", manifest["tokenization"]["total_tokens"],
      "registered sources", [x["name"] for x in registry.list()])


In [ ]:
from pathlib import Path
from training.recovery import list_valid_checkpoints, transfer_checkpoint
from training.config import TrainConfig
from training.trainer import build_from_dirs
from model.config import load_config
import json
OUTPUT = Path('/kaggle/working/minicoder_output')
CHECKPOINTS = OUTPUT / 'checkpoints'; CHECKPOINTS.mkdir(parents=True, exist_ok=True)
# If a previous committed Kaggle Output was attached as an input Dataset, import
# the newest checkpoint only after its checksum/manifest verifies.
if not list_valid_checkpoints(CHECKPOINTS):
    candidates = set(Path('/kaggle/input').rglob('checkpoint_step_*.pt'))
    valid_sources = []
    for candidate in candidates:
        try:
            valid_sources.extend(list_valid_checkpoints(candidate.parent))
        except OSError:
            pass
    if valid_sources:
        source = max(valid_sources, key=lambda item: item['step'])
        print('Import verified checkpoint:', source['path'])
        transfer_checkpoint(source['path'], CHECKPOINTS)
valid = list_valid_checkpoints(CHECKPOINTS)
next_target = (valid[0]['step'] + 2) if valid else 8
run_dir = OUTPUT / 'run'
tcfg = TrainConfig(data_dir=str(data), run_dir=str(run_dir), checkpoint_dir=str(CHECKPOINTS),
                   seq_len=64, micro_batch_size=2, grad_accum=2, max_steps=next_target,
                   warmup_steps=2, eval_every=4, eval_batches=4, save_every=2, log_every=2,
                   learning_rate=0.003, device='auto', resume=bool(valid))
result = build_from_dirs(load_config('config/model_tiny_10m.yaml'), tcfg).train()
assert result['final_step'] == next_target and result['final_eval'] is not None
print(json.dumps(result, indent=2))

In [ ]:
# Reconstruct the Trainer from the last portable checkpoint, advance two more steps, export.
from training.recovery import list_valid_checkpoints
from training.config import TrainConfig
from training.trainer import build_from_dirs
from model.config import load_config
prior = list_valid_checkpoints(CHECKPOINTS)
assert prior, 'No valid local or imported checkpoint'
target = prior[0]['step'] + 2
resume_cfg = TrainConfig(**{**tcfg.__dict__, 'max_steps': target, 'resume': True})
resumed = build_from_dirs(load_config('config/model_tiny_10m.yaml'), resume_cfg).train()
assert resumed['final_step'] == target and resumed['tokens_seen'] > result['tokens_seen']
from model.export_cli import export_checkpoint
latest = list_valid_checkpoints(CHECKPOINTS)[0]
export_result = export_checkpoint(latest['path'], str(OUTPUT / 'export' / 'MiniCoder-Tiny'))
shutil.copy2(run_dir / 'metrics.jsonl', OUTPUT / 'metrics.jsonl')
print('Resume:', json.dumps(resumed, indent=2))
print('Model export:', export_result)
print('Commit/save this Notebook output. For a new session, attach the prior Output Dataset as input; the next run imports its verified checkpoint.')

## Limits

`/kaggle/working` is not assumed permanent until the user commits/saves Notebook output. External controller submission is not implemented; this code runs only after the user starts the Kaggle Notebook. Cross-session restore is implemented by importing a previously saved output checkpoint with integrity verification. Public dataset use depends on internet settings and user-verified license. No real Kaggle account/GPU execution has been run in this sandbox.